# Logistic Regression Classification Across Four Datasets
**Datasets:** Candy Rankings · Titanic Survival · Breast Cancer Wisconsin · Pima Indians Diabetes  
**Methods:** Logistic Regression · Feature Engineering · Class Imbalance Handling · Model Evaluation

---

## Overview

Logistic regression is a foundational classification algorithm that models the probability of a binary outcome using a sigmoid (logistic) function applied to a linear combination of input features:

$$p = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + ... + \beta_n x_n)}}$$

**Properties:**
- The dependent variable is assumed to follow a Bernoulli distribution (binary outcome)
- Parameters are estimated via Maximum Likelihood Estimation (MLE)

**Advantages:**
- Simple, fast to train, and doesn't require heavy compute
- Coefficients are directly interpretable as log-odds contributions
- Outputs calibrated probabilities, not just hard class labels

This notebook applies logistic regression to four datasets of increasing complexity, each requiring a different approach to feature engineering, class imbalance, and evaluation:

1. **Candy Rankings** — a simple introductory example predicting whether a candy contains chocolate
2. **Titanic Survival** — feature engineering and categorical encoding
3. **Breast Cancer Wisconsin** — correlated features and model comparison
4. **Pima Indians Diabetes** — class imbalance and handling biologically impossible zero values

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, log_loss

## 2. Introductory Example — Candy Rankings

We start with a simple binary classification task: predicting whether a candy contains chocolate based on its other characteristics (fruity, caramel, nougat, sugar percentile, win percentile, etc.). This dataset is small and clean, making it ideal for walking through the core sklearn logistic regression workflow before tackling messier real-world data.

In [ ]:
# Download the FiveThirtyEight candy rankings dataset
!wget -q https://raw.githubusercontent.com/fivethirtyeight/data/master/candy-power-ranking/candy-data.csv

In [ ]:
df_candy = pd.read_csv('candy-data.csv')

candy_features = ['fruity', 'caramel', 'peanutyalmondy', 'nougat',
                   'crispedricewafer', 'hard', 'bar', 'pluribus',
                   'sugarpercent', 'pricepercent', 'winpercent', 'chocolate']
df_candy = df_candy[candy_features]

print(f"Shape: {df_candy.shape}")
df_candy.head()

### 2.1 Train/test split and model fitting

In [ ]:
feature_cols_candy = ['fruity', 'caramel', 'peanutyalmondy', 'nougat',
                      'crispedricewafer', 'hard', 'bar', 'pluribus',
                      'sugarpercent', 'pricepercent', 'winpercent']

train_candy, test_candy = train_test_split(df_candy, test_size=0.2, random_state=42)

X_train_candy = train_candy[feature_cols_candy]
y_train_candy = train_candy['chocolate']
X_test_candy  = test_candy[feature_cols_candy]
y_test_candy  = test_candy['chocolate']

model_candy = LogisticRegression(max_iter=1000)
model_candy.fit(X_train_candy, y_train_candy)

print("Coefficients:")
for feat, coef in zip(feature_cols_candy, model_candy.coef_[0]):
    print(f"  {feat:<20}: {coef:>8.3f}")
print(f"  {'intercept':<20}: {model_candy.intercept_[0]:>8.3f}")

**Interpreting the coefficients:** positive coefficients increase the predicted log-odds of a candy containing chocolate; negative coefficients decrease it. Features like `bar` and `peanutyalmondy` are typically among the strongest positive predictors, since chocolate bars and peanut/almond candies are commonly chocolate-based.

In [ ]:
# Predict on the test set
y_pred_candy  = model_candy.predict(X_test_candy)
y_probs_candy = model_candy.predict_proba(X_test_candy)

print("Predictions:  ", y_pred_candy)
print("Probabilities (first 5):")
print(y_probs_candy[:5].round(3))

### 2.2 Evaluation

In [ ]:
acc  = metrics.accuracy_score(y_test_candy, y_pred_candy)
f1   = metrics.f1_score(y_test_candy, y_pred_candy)
rec  = metrics.recall_score(y_test_candy, y_pred_candy)
cm   = metrics.confusion_matrix(y_test_candy, y_pred_candy)

print(f"Accuracy: {acc:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Recall:   {rec:.3f}")
print(f"\nConfusion Matrix:\n{cm}")
print("\n(Rows = actual class, Columns = predicted class)")

### 2.3 Scaling features

Logistic regression coefficients are sensitive to feature scale — a feature ranging 0–100 will dominate one ranging 0–1 unless scaled. `StandardScaler` standardizes features to zero mean and unit variance. The cleanest way to combine scaling and modeling is `make_pipeline`, which ensures the scaler is fit only on training data and applied consistently to test data.

In [ ]:
model_candy_scaled = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_candy_scaled.fit(X_train_candy, y_train_candy)

y_pred_candy_scaled = model_candy_scaled.predict(X_test_candy)
acc_scaled = metrics.accuracy_score(y_test_candy, y_pred_candy_scaled)

print(f"Accuracy without scaling: {acc:.3f}")
print(f"Accuracy with scaling:    {acc_scaled:.3f}")
print("\n(For this dataset scaling has limited impact since most features are")
print(" already 0/1 binary flags — scaling matters more when features span")
print(" very different numeric ranges, as in the datasets below.)")

## 3. Titanic Survival Prediction

The Titanic dataset is a classic benchmark for classification: predicting passenger survival based on demographic and travel information. Unlike the candy dataset, this one requires real feature engineering — handling missing data, encoding categorical variables, and constructing new features from existing ones.

In [ ]:
df_titanic = sns.load_dataset('titanic')
print(f"Shape: {df_titanic.shape}")
df_titanic.head()

In [ ]:
# Check for missing data
print("Missing values per column:")
print(df_titanic.isnull().sum().sort_values(ascending=False).head(8))

### 3.1 Feature engineering

We construct several derived features that are known to be predictive of Titanic survival:
- **family_size** — total family members aboard (siblings/spouses + parents/children + self)
- **is_alone** — binary flag for solo travelers, who had notably different survival rates
- **title** — derived from the `who` field (man/woman/child) as a simplified honorific proxy

In [ ]:
df_titanic['family_size'] = df_titanic['sibsp'] + df_titanic['parch'] + 1
df_titanic['is_alone']    = (df_titanic['family_size'] == 1).astype(int)
df_titanic['title']       = df_titanic['who'].map({'man': 'Mr', 'woman': 'Mrs', 'child': 'Master'})

print("New features:")
df_titanic[['family_size', 'is_alone', 'title']].head()

### 3.2 Handling missing values

Rather than dropping rows with missing data (which would discard useful information), we impute:
- **age** — filled with the median age within each title group (children, men, and women have different typical ages)
- **fare** — filled with the median fare within each passenger class
- **embark_town** — filled with the most common embarkation port (mode)

In [ ]:
df_titanic['age']         = df_titanic.groupby('title')['age'].transform(lambda x: x.fillna(x.median()))
df_titanic['fare']        = df_titanic.groupby('pclass')['fare'].transform(lambda x: x.fillna(x.median()))
df_titanic['embark_town'] = df_titanic['embark_town'].fillna(df_titanic['embark_town'].mode()[0])

print("Missing values remaining in our feature columns:")
print(df_titanic[['age', 'fare', 'embark_town']].isnull().sum())

### 3.3 Encode categorical variables and split

`pd.get_dummies` converts categorical columns (sex, embark_town, title) into binary indicator columns. `drop_first=True` avoids the dummy variable trap by dropping one category per feature (its effect is captured in the intercept).

In [ ]:
FEATURES = ['pclass', 'sex', 'age', 'fare', 'embark_town', 'family_size', 'is_alone', 'title']
TARGET   = 'survived'

data_titanic = df_titanic[FEATURES + [TARGET]].copy()
data_titanic = pd.get_dummies(data_titanic, columns=['sex', 'embark_town', 'title'], drop_first=True)

print(f"Final feature columns: {list(data_titanic.columns)}")

In [ ]:
X = data_titanic.drop(columns=[TARGET])
y = data_titanic[TARGET]

# Stratify ensures train and test sets have the same survival rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train survival rate: {y_train.mean():.2%}")
print(f"Test survival rate:  {y_test.mean():.2%}")

### 3.4 Fit and evaluate the model

In [ ]:
model_titanic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_titanic.fit(X_train, y_train)

lr = model_titanic.named_steps['logisticregression']
print("Coefficients:")
for feat, coef in zip(X.columns, lr.coef_[0]):
    print(f"  {feat:<20}: {coef:>8.3f}")
print(f"  {'intercept':<20}: {lr.intercept_[0]:>8.3f}")

In [ ]:
y_pred_titanic = model_titanic.predict(X_test)

acc_titanic = metrics.accuracy_score(y_test, y_pred_titanic)
cm_titanic  = metrics.confusion_matrix(y_test, y_pred_titanic)

print(f"Accuracy: {acc_titanic:.3f}")
print(f"\nConfusion Matrix:\n{cm_titanic}")

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_titanic, display_labels=['Died', 'Survived']).plot(ax=ax, cmap='Blues')
ax.set_title('Titanic — Confusion Matrix')
plt.tight_layout()
plt.show()

**Interpretation:** sex and passenger class are typically the strongest predictors of survival on the Titanic — consistent with the historical "women and children first" evacuation policy and the fact that first-class cabins were closer to lifeboats.

## 4. Breast Cancer Wisconsin — Tumor Classification

This dataset contains measurements computed from digitized images of breast mass biopsies, used to classify tumors as malignant or benign. Unlike Titanic, the features here are all continuous and **highly correlated** with each other (e.g., radius, perimeter, and area all measure related aspects of cell size), which makes feature selection and model comparison particularly useful.

**Note:** in this sklearn dataset, `target = 0` is malignant and `target = 1` is benign — easy to get backwards.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df_cancer = pd.DataFrame(data.data, columns=data.feature_names)
df_cancer['target'] = data.target

print(f"Shape: {df_cancer.shape}")
print(f"Class balance:\n{df_cancer['target'].value_counts()}")
print(f"\n0 = malignant, 1 = benign")

### 4.1 Explore feature correlation

Many of these features measure overlapping aspects of cell morphology. A correlation heatmap of a representative subset shows just how related some features are — useful for deciding which to keep versus drop.

In [ ]:
# Look at correlation among the "mean" measurement features
mean_features = [c for c in df_cancer.columns if c.startswith('mean')][:10]

plt.figure(figsize=(10, 8))
sns.heatmap(df_cancer[mean_features].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature Correlation — "Mean" Measurements')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize how well a couple of features separate the two classes
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, feat in zip(axes, ['mean radius', 'mean texture']):
    for label, name, color in [(0, 'Malignant', 'crimson'), (1, 'Benign', 'steelblue')]:
        subset = df_cancer[df_cancer['target'] == label]
        ax.hist(subset[feat], bins=25, alpha=0.6, label=name, color=color)
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
    ax.legend()
    ax.set_title(f'{feat} by Class')

plt.tight_layout()
plt.show()

### 4.2 Train/test split and scaling

With 30 continuous features on very different scales (e.g., `mean area` can be in the hundreds while `mean smoothness` is around 0.1), scaling is essential.

In [ ]:
X_cancer = df_cancer.drop(columns=['target'])
y_cancer = df_cancer['target']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

print(f"Training samples: {len(X_train_c)}")
print(f"Test samples:     {len(X_test_c)}")

### 4.3 Compare multiple models

In [ ]:
models_cancer = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    'KNN':                 make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
}

print(f"{'Model':<22} {'Accuracy':>10} {'F1':>10} {'Recall':>10}")
print("-" * 54)

results_cancer = {}
for name, model in models_cancer.items():
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    acc = metrics.accuracy_score(y_test_c, preds)
    f1  = metrics.f1_score(y_test_c, preds)
    rec = metrics.recall_score(y_test_c, preds)
    results_cancer[name] = preds
    print(f"{name:<22} {acc:>10.3f} {f1:>10.3f} {rec:>10.3f}")

**Why recall matters most here:** in a cancer screening context, a false negative (predicting benign when the tumor is actually malignant) is far more costly than a false positive. Recall on the malignant class is arguably more important than overall accuracy.

In [ ]:
# Confusion matrix for the best-performing model (typically Logistic Regression
# or Random Forest on this well-separated dataset)
best_model_name = max(results_cancer, key=lambda k: metrics.f1_score(y_test_c, results_cancer[k]))
cm_cancer = metrics.confusion_matrix(y_test_c, results_cancer[best_model_name])

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_cancer, display_labels=['Malignant', 'Benign']).plot(ax=ax, cmap='Reds')
ax.set_title(f'Breast Cancer — Confusion Matrix ({best_model_name})')
plt.tight_layout()
plt.show()

print(f"Best model by F1 score: {best_model_name}")

## 5. Pima Indians Diabetes — Class Imbalance and Data Quality Issues

This dataset predicts diabetes diagnosis from medical measurements. It illustrates two common real-world data problems:

1. **Class imbalance** — there are noticeably more non-diabetic than diabetic cases
2. **Biologically impossible zero values** — features like glucose, blood pressure, and BMI contain zeros that represent *missing* measurements, not true zero values (a blood glucose level of 0 is not physiologically possible)

In [ ]:
from sklearn.datasets import fetch_openml

X_diabetes, y_diabetes = fetch_openml("diabetes", version=1, return_X_y=True, as_frame=True)

print(f"Shape: {X_diabetes.shape}")
print(f"Features: {list(X_diabetes.columns)}")
print(f"\nClass balance:\n{y_diabetes.value_counts()}")

In [ ]:
# Encode the target as binary (tested_positive = 1, tested_negative = 0)
y_diabetes_bin = (y_diabetes == 'tested_positive').astype(int)

print(f"Positive rate: {y_diabetes_bin.mean():.1%}")
print("This is a moderately imbalanced dataset — about 1 in 3 patients tested positive")

### 5.1 Identify and fix impossible zero values

Glucose, blood pressure, skin thickness, insulin, and BMI cannot legitimately be zero in a living patient. These zeros are almost certainly missing data encoded as 0 rather than NaN — a common data quality issue in older medical datasets.

In [ ]:
zero_invalid_cols = ['plas', 'pres', 'skin', 'insu', 'mass']  # glucose, BP, skin, insulin, BMI

print("Percentage of zero values in features where zero is biologically implausible:")
for col in zero_invalid_cols:
    if col in X_diabetes.columns:
        pct_zero = (X_diabetes[col] == 0).mean()
        print(f"  {col:<8}: {pct_zero:.1%}")

In [ ]:
# Replace implausible zeros with NaN, then impute with the column median
X_diabetes_clean = X_diabetes.copy()

for col in zero_invalid_cols:
    if col in X_diabetes_clean.columns:
        X_diabetes_clean[col] = X_diabetes_clean[col].replace(0, np.nan)
        X_diabetes_clean[col] = X_diabetes_clean[col].fillna(X_diabetes_clean[col].median())

print("Zero values remaining after cleaning:")
for col in zero_invalid_cols:
    if col in X_diabetes_clean.columns:
        print(f"  {col:<8}: {(X_diabetes_clean[col] == 0).sum()}")

### 5.2 Train/test split and model fitting

In [ ]:
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diabetes_clean, y_diabetes_bin, test_size=0.2, random_state=42, stratify=y_diabetes_bin
)

model_diabetes = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_diabetes.fit(X_train_d, y_train_d)

y_pred_diabetes  = model_diabetes.predict(X_test_d)
y_probs_diabetes = model_diabetes.predict_proba(X_test_d)

acc_d = metrics.accuracy_score(y_test_d, y_pred_diabetes)
f1_d  = metrics.f1_score(y_test_d, y_pred_diabetes)
rec_d = metrics.recall_score(y_test_d, y_pred_diabetes)

print(f"Accuracy: {acc_d:.3f}")
print(f"F1 Score: {f1_d:.3f}")
print(f"Recall:   {rec_d:.3f}")

### 5.3 Compare against an imbalance-aware model

`class_weight='balanced'` automatically reweights the loss function to penalize misclassifying the minority class more heavily — a simple, effective way to handle imbalance without resampling.

In [ ]:
model_diabetes_balanced = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight='balanced')
)
model_diabetes_balanced.fit(X_train_d, y_train_d)
y_pred_balanced = model_diabetes_balanced.predict(X_test_d)

print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'Recall':>10}")
print("-" * 57)
print(f"{'Standard':<25} {acc_d:>10.3f} {f1_d:>10.3f} {rec_d:>10.3f}")
print(f"{'Class-weighted':<25}"
      f"{metrics.accuracy_score(y_test_d, y_pred_balanced):>10.3f}"
      f"{metrics.f1_score(y_test_d, y_pred_balanced):>10.3f}"
      f"{metrics.recall_score(y_test_d, y_pred_balanced):>10.3f}")

print("\nClass weighting typically trades a small amount of accuracy for")
print("substantially improved recall on the positive (diabetic) class —")
print("an important tradeoff in a medical screening context.")

## 6. Summary

This notebook applied logistic regression to four datasets, each highlighting a different practical challenge:

| Dataset | Key challenge | Technique applied |
|---|---|---|
| **Candy Rankings** | None (introductory) | Basic sklearn workflow, coefficient interpretation |
| **Titanic** | Missing data, categorical features | Group-wise imputation, one-hot encoding, derived features |
| **Breast Cancer** | Highly correlated features | Correlation analysis, multi-model comparison |
| **Diabetes** | Class imbalance, invalid zero values | Zero-as-missing detection, median imputation, `class_weight='balanced'` |

**Key takeaways:**
- Always scale features before logistic regression when they're on different numeric scales
- Missing data should be imputed thoughtfully (group-wise medians, not blanket fills) rather than dropped outright
- In medical or safety-critical classification tasks, recall on the positive class often matters more than overall accuracy — a missed diagnosis is typically far costlier than a false alarm
- `class_weight='balanced'` is a simple, effective first step for handling imbalanced classes before reaching for more complex resampling techniques
- Domain knowledge matters: recognizing that a glucose reading of 0 is impossible, not just "low," is what separates a naive model from a clinically sound one